# Tree-of-Thought 实战：24点游戏

用4个数字，通过加减乘除运算得到24。

**思路（Tree-of-Thought）：**
1. **提议（Propose）**：列出所有可能的两两运算
2. **评估（Value）**：评估剩余数字达到24的可能性
3. **选择（Select）**：保留最有希望的方案，继续下一步
4. 重复直到找到答案

In [7]:
import os, re
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

def qwen_call(prompt, model="qwen-plus", n=1):
    """调用通义千问API，模拟多次采样（n次独立调用）"""
    outputs = []
    for _ in range(n):
        res = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        outputs.append(res.choices[0].message.content)
    return outputs

print("API 初始化完成")

API 初始化完成


## 定义提示词模板

- **propose_prompt**：让模型列出所有可能的两两运算
- **value_prompt**：让模型评估剩余数字能否凑到24

In [8]:
propose_prompt = """给定数字，列出所有可能的两两运算（加、减、乘、除），每行一个。
格式：a [运算符] b = 结果（剩余：其他数字和结果）

示例输入：2 8 14
可能的下一步：
2 + 8 = 10（剩余：10 14）
8 - 2 = 6（剩余：6 14）
2 * 8 = 16（剩余：16 14）
8 / 2 = 4（剩余：4 14）
2 + 14 = 16（剩余：16 8）
14 - 2 = 12（剩余：12 8）
2 * 14 = 28（剩余：28 8）
14 / 2 = 7（剩余：7 8）
8 + 14 = 22（剩余：22 2）
14 - 8 = 6（剩余：6 2）
8 * 14 = 112（剩余：112 2）
14 / 8 = 1.75（剩余：1.75 2）

输入：{input}
可能的下一步：
"""

value_prompt = """评估给定的数字是否可以通过加减乘除运算达到24，输出sure(20)/likely(1)/impossible(0.001)，只输出分值。

10 14
10 + 14 = 24
left: 20
11 12
11 + 12 = 23
12 - 11 = 1
11 * 12 = 132
11 / 12 = 0.91
left: 0.001
4 4 10
4 + 4 + 10 = 8 + 10 = 18
4 * 10 - 4 = 40 - 4 = 36
(10 - 4) * 4 = 6 * 4 = 24
left: 20
4 9 11
9 + 11 + 4 = 20 + 4 = 24
left: 20
5 7 8
5 + 7 + 8 = 12 + 8 = 20
(8 - 5) * 7 = 3 * 7 = 21
无法达到24，但数字在合理范围内
left: 1
5 6 6
5 + 6 + 6 = 17
(6 - 5) * 6 = 6
5 * 6 - 6 = 24
left: 20
10 10 10
10 + 10 + 10 = 30
10 * 10 - 10 = 90
(10 - 10) * 10 = 0
无法达到24
left: 0.001
1 3 3
1 + 3 + 3 = 7
1 * 3 * 3 = 9
(3 - 1) * 3 = 6
3 / 1 * 3 = 9
1 * 3 + 3 = 6
无法达到24
left: 0.001

{input}
"""

print("提示词模板定义完成")

提示词模板定义完成


## 核心函数

- `propose()` — 提议：列出所有可能的运算
- `evaluate()` — 评估：打分判断可行性
- `solve_game24()` — 求解：beam search 逐层搜索

In [9]:
def is_number(s):
    """检查字符串是否为有效数字"""
    try:
        float(s)
        return True
    except ValueError:
        return False

def propose(numbers_str):
    """第1步：提议 - 让模型列出所有可能的两两运算"""
    prompt = propose_prompt.format(input=numbers_str)
    response = qwen_call(prompt)[0]
    print(f"\n[propose 原始响应]\n{response}\n")
    proposals = []
    for line in response.strip().split('\n'):
        line = line.strip().lstrip('- ')
        if '剩余' in line and '=' in line:
            remaining = parse_remaining(line)
            if remaining and all(is_number(x) for x in remaining.split()):
                proposals.append(line)
    return proposals

def parse_remaining(proposal):
    """从提议中解析剩余数字，如 '2 + 8 = 10（剩余：10 14）' -> '10 14'"""
    match = re.search(r'剩余[：:]?\s*([^）\)]+)', proposal)
    if match:
        return match.group(1).strip()
    return None

def evaluate(remaining_str):
    """第2步：评估 - 判断剩余数字达到24的可能性"""
    prompt = value_prompt.format(input=remaining_str)
    response = qwen_call(prompt)[0]
    print(f"[evaluate {remaining_str}] 响应: {response.strip().split(chr(10))[-1]}")
    for line in reversed(response.strip().split('\n')):
        match = re.search(r'(\d+\.?\d*)', line)
        if match:
            val = float(match.group(1))
            if val in (20, 1, 0.001):
                return val
    return 0.001

def solve_game24(numbers, beam_width=3, max_depth=3):
    """
    Tree-of-Thought 求解24点
    beam_width: 每层保留最优方案数
    max_depth: 最大搜索深度
    """
    current_states = [(' '.join(map(str, numbers)), [])]

    for depth in range(max_depth):
        print(f"\n{'='*50}")
        print(f"第 {depth + 1} 步搜索")
        print(f"{'='*50}")

        candidates = []

        for state, history in current_states:
            nums = state.split()
            if len(nums) <= 1:
                continue

            proposals = propose(state)
            print(f"\n输入数字: {state}")
            print(f"有效提议数量: {len(proposals)}")

            for p in proposals:
                remaining = parse_remaining(p)
                if remaining is None:
                    continue

                score = evaluate(remaining)
                candidates.append((remaining, history + [p], score))
                print(f"  {p} -> 评分: {score}")

        if not candidates:
            print("没有找到有效的候选方案")
            break

        candidates.sort(key=lambda x: x[2], reverse=True)
        current_states = [(c[0], c[1]) for c in candidates[:beam_width]]

        print(f"\n保留前 {beam_width} 个最优方案:")
        for state, history, score in candidates[:beam_width]:
            print(f"  剩余: {state}, 评分: {score}")

        for state, history, score in candidates[:beam_width]:
            nums = state.split()
            if len(nums) == 1 and is_number(nums[0]) and abs(float(nums[0]) - 24) < 0.001:
                print(f"\n找到答案！步骤:")
                for i, step in enumerate(history):
                    print(f"  第{i+1}步: {step}")
                return history

    print("\n未能在限定步数内找到确定的答案")
    return None

print("核心函数定义完成")

核心函数定义完成


## 运行：用 4, 5, 6, 10 凑24

In [11]:
result = solve_game24([4, 5, 6, 10])


第 1 步搜索

输入数字: 4 5 6 10
有效提议数量: 136
  4 + 5 = 9（剩余：9 6 10） -> 评分: 20.0
  4 × 5 = 20（剩余：20 6 10） -> 评分: 20.0
  4 ÷ 5 = 0.8（剩余：0.8 6 10） -> 评分: 1.0
  4 + 6 = 10（剩余：10 5 10）→ 注意：剩余含两个10（原始10仍在！），合法（数字可重复） -> 评分: 20.0
  4 × 6 = 24（剩余：24 5 10） -> 评分: 20.0
  4 + 6 = 10（剩余：10 5 10） -> 评分: 1.0
  4 × 6 = 24（剩余：24 5 10） -> 评分: 20.0
  4 + 10 = 14（剩余：14 5 6） -> 评分: 20.0
  4 × 10 = 40（剩余：40 5 6） -> 评分: 20.0
  4 ÷ 10 = 0.4（剩余：0.4 5 6） -> 评分: 1.0
  5 + 4 = 9（剩余：9 6 10） -> 评分: 20.0
  5 − 4 = 1（剩余：1 6 10） -> 评分: 20.0
  5 × 4 = 20（剩余：20 6 10） -> 评分: 20.0
  5 ÷ 4 = 1.25（剩余：1.25 6 10） -> 评分: 1.0
  5 + 6 = 11（剩余：11 4 10） -> 评分: 20.0
  5 × 6 = 30（剩余：30 4 10） -> 评分: 20.0
  5 + 10 = 15（剩余：15 4 6） -> 评分: 20.0
  5 × 10 = 50（剩余：50 4 6） -> 评分: 1.0
  5 ÷ 10 = 0.5（剩余：0.5 4 6） -> 评分: 1.0
  6 + 4 = 10（剩余：10 5 10）← 同前，但这是不同有序对，仍列出 -> 评分: 1.0
  6 − 4 = 2（剩余：2 5 10） -> 评分: 1.0
  6 × 4 = 24（剩余：24 5 10） -> 评分: 20.0
  6 ÷ 4 = 1.5（剩余：1.5 5 10） -> 评分: 1.0
  6 + 5 = 11（剩余：11 4 10） -> 评分: 20.0
  6 − 5 = 1（剩余：1 4 10） -> 评分: 20